# EU Commission (Funding & Tenders Portal — Calls for Tenders)

Fetches live "Calls for tenders" from the European Commission's Funding & Tenders Portal (SEDIA search API), filters to the target CPV codes, and uploads new contracts to the unified Notion database.

**Link:** https://ec.europa.eu/info/funding-tenders/opportunities/portal/screen/opportunities/calls-for-tenders?isExactMatch=true&order=DESC&pageNumber=1&pageSize=50&sortBy=startDate

**Filters applied:**
- Procedure type: Open procedure, Call for expression of interest (both variants), Planned negotiation procedure, Accelerated open procedure
- Submission status: Forthcoming + Open
- CPV codes: same 27-code list used across all sources, matched client-side against `mainCpv`
- Blocked keywords: notices are hard-excluded if the title or description matches any term in `blocked_words.py` (shared blocklist, sources/ root) - see that file for the current list and notes on match behaviour

In [1]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]

DATABASE_ID = '334701e728cb8096a94cebc0985684a2'

headers_notion = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}


In [2]:
# Shared keyword blocklist (sources/ root) - added 2026-08-09 per Javiera's feedback
import sys
from pathlib import Path

BLOCKED_WORDS_PATH = Path("../blocked_words.py")
if not BLOCKED_WORDS_PATH.exists():
    raise FileNotFoundError(f"Could not find {BLOCKED_WORDS_PATH.resolve()}")

sys.path.insert(0, str(BLOCKED_WORDS_PATH.parent.resolve()))
from blocked_words import is_blocked, blocked_keyword_hits


In [3]:
import requests
import pandas as pd
import json
import html
import os
import time
from datetime import datetime, timezone
from dateutil import parser as _dateparser

FT_API_URL = "https://api.tech.ec.europa.eu/search-api/prod/rest/search"
FT_API_KEY = "SEDIA"

# Confirmed via live capture: Open procedure, both Call for expression of interest
# variants, Planned negotiation procedure, Accelerated open procedure
PROCEDURE_TYPE_CODES = ["47396220", "47396202", "47396204", "47396214", "47396198"]

# Confirmed via live capture: Forthcoming + Open (Closed = 31094503, excluded)
STATUS_CODES = ["31094501", "31094502"]

# Same 27-code consultancy/research CPV list used across all sources
TARGET_CPV = {
    "66171000", "73000000", "73100000", "73110000", "73120000", "73200000",
    "73210000", "73220000", "73300000", "73400000", "75210000", "75211200",
    "79311100", "79311300", "79311400", "79311410", "79313000", "79314000",
    "79315000", "79320000", "79330000", "79411000", "79411100", "79419000",
    "90713000", "98200000",
}

DISPLAY_FIELDS = [
    "title", "description", "mainCpv", "cftEstimatedTotalProcedureValue",
    "cftEstimatedOverallContractCurrency", "deadlineDate", "startDate",
    "cftLeadContractingAuthorityCode", "url", "callIdentifier", "cftId",
    "status", "procedureType", "closingDate",
]


def clean_description(description):
    if not description:
        return "Not Disclosed"
    cleaned = html.unescape(description)
    cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
    return ' '.join(cleaned.split()).strip()


def parse_buyer_name(raw_authority_field) -> str:
    """
    cftLeadContractingAuthorityCode comes back as a JSON-encoded string like:
    '[{"name":"European Union Agency for Cybersecurity (ENISA)","link":"...","isLeadAuthority":true}]'
    NOTE: 'caName' looked unreliable in testing (came back equal to the tender
    title rather than an authority name) - deliberately not used here.
    """
    if not raw_authority_field:
        return "Not Disclosed"
    try:
        authorities = json.loads(raw_authority_field[0]) if isinstance(raw_authority_field, list) else json.loads(raw_authority_field)
        if not authorities:
            return "Not Disclosed"
        lead = next((a for a in authorities if a.get("isLeadAuthority")), authorities[0])
        return (lead.get("name") or "Not Disclosed").strip() or "Not Disclosed"
    except Exception:
        return "Not Disclosed"


def parse_buyer_link(raw_authority_field) -> str:
    try:
        authorities = json.loads(raw_authority_field[0]) if isinstance(raw_authority_field, list) else json.loads(raw_authority_field)
        if not authorities:
            return ""
        lead = next((a for a in authorities if a.get("isLeadAuthority")), authorities[0])
        return lead.get("link") or ""
    except Exception:
        return ""


def fetch_ft_page(page_num: int, page_size: int = 50) -> dict:
    query = {"bool": {"must": [
        {"terms": {"type": ["0"]}},
        {"terms": {"DATASOURCE": ["SEDIA"]}},
        {"terms": {"procedureType": PROCEDURE_TYPE_CODES}},
        {"terms": {"status": STATUS_CODES}},
        {"terms": {"language": ["en"]}},
    ]}}
    sort = {"field": "startDate", "order": "DESC"}

    for attempt in range(5):
        try:
            resp = requests.post(
                FT_API_URL,
                params={
                    "apiKey": FT_API_KEY,
                    "text": "***",
                    "pageSize": str(page_size),
                    "pageNumber": str(page_num),
                },
                files={
                    "query": ("blob", json.dumps(query), "application/json"),
                    "sort": ("blob", json.dumps(sort), "application/json"),
                    "languages": ("blob", json.dumps(["en"]), "application/json"),
                    "displayFields": ("blob", json.dumps(DISPLAY_FIELDS), "application/json"),
                },
                timeout=30,
            )
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                time.sleep(2 ** attempt)
            else:
                print(f"\u26a0\ufe0f Unexpected status {resp.status_code} on page {page_num}: {resp.text[:300]}")
                break
        except requests.exceptions.RequestException as e:
            print(f"\u26a0\ufe0f Request error on page {page_num}: {e}")
            time.sleep(2 ** attempt)
    return {}


In [4]:
# 1) Load already-uploaded titles to avoid duplicates
csv_path = "eu_commission_contract_titles.csv"
existing_titles = set()
if os.path.exists(csv_path):
    try:
        prev = pd.read_csv(csv_path)
        if "Title" in prev.columns:
            existing_titles = set(prev["Title"].dropna().astype(str).str.strip().str.lower())
    except Exception as e:
        print("Warning: could not read", csv_path, ":", e)

# 2) Fetch + filter + parse
extracted_data = []
page = 1
MAX_PAGES = 40  # safety cap; same as the OppsLink notebook

while page <= MAX_PAGES:
    data = fetch_ft_page(page)
    results = data.get("results", [])
    if not results:
        break

    for item in results:
        m = item.get("metadata", {})

        title_list = m.get("title") or []
        title = (title_list[0] if title_list else "").strip()
        if not title:
            continue

        if title.strip().lower() in existing_titles:
            continue

        cpv_list = m.get("mainCpv") or []
        if not (set(cpv_list) & TARGET_CPV):
            continue  # client-side CPV filter

        closing_date_list = m.get("closingDate") or []
        deadline_list = m.get("deadlineDate") or []
        closing_date = (closing_date_list[0] if closing_date_list else None) or (deadline_list[0] if deadline_list else None)

        value_list = m.get("cftEstimatedTotalProcedureValue") or []
        # already includes currency, e.g. "600000 EUR" - do not append currency again
        value = value_list[0] if value_list else "Unavailable"

        desc_list = m.get("description") or []
        description = clean_description(desc_list[0] if desc_list else "")

        url_list = m.get("url") or []
        link = url_list[0] if url_list else ""

        buyer_name = parse_buyer_name(m.get("cftLeadContractingAuthorityCode"))
        buyer_link = parse_buyer_link(m.get("cftLeadContractingAuthorityCode"))

        if is_blocked(title, description):
            hits = blocked_keyword_hits(title, description)
            print(f"⛔ Skipping blocked keyword ({', '.join(hits)}): {title}")
            continue

        extracted_data.append({
            "closing_date": closing_date,
            "country": "EU",  # no confirmed location field on this source - same fallback as OppsLink notebook
            "client": buyer_name,
            "client_link": buyer_link,
            "link": link,
            "title": title,
            "description": description,
            "value": value,
            "cpv_codes": ", ".join(cpv_list),
            "language": "English",
        })

    total_results = data.get("totalResults", 0)
    print(f"Page {page} -> {len(results)} hits (running total after filters: {len(extracted_data)} / {total_results} raw results)")

    if len(results) < 50:
        break
    page += 1
    time.sleep(1)

print(f"\u2705 {len(extracted_data)} new contracts ready for Notion upload")


⛔ Skipping blocked keyword (operational): EU Complementary Time (C-T) and Complementary Position and Navigation (C-PN) services
Page 1 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 2 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 3 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 4 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 5 -> 50 hits (running total after filters: 0 / 596 raw results)


...

Page 8 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 9 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 10 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 11 -> 50 hits (running total after filters: 0 / 596 raw results)


Page 12 -> 46 hits (running total after filters: 0 / 596 raw results)
✅ 0 new contracts ready for Notion upload


### Upload to Notion

In [5]:
def create_page(properties: dict) -> bool:
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": DATABASE_ID}, "properties": properties}
    res = requests.post(url, headers=headers_notion, json=payload, timeout=60)
    if not res.ok:
        print("\u274c Notion error:", res.status_code, res.text[:500])
        return False
    print(f"\u2705 Page created: {properties['Name']['title'][0]['text']['content']}")
    return True


def _safe_str(x, default=""):
    if x is None:
        return default
    s = str(x).strip()
    return s if s else default


def _safe_iso(dt_str):
    if not dt_str:
        return None
    try:
        parsed = _dateparser.parse(dt_str)
        return parsed.isoformat()
    except Exception:
        return None


now_iso = datetime.now(timezone.utc).isoformat()
new_titles_for_csv = []

# Upload newest first, matching Find_tender_notion.ipynb's convention
for contract in list(reversed(extracted_data)):
    name = _safe_str(contract.get("title"))[:1000]
    if not name:
        continue

    closing_date_iso = _safe_iso(contract.get("closing_date"))

    props = {
        "Name": {"title": [{"text": {"content": name}}]},
        "CPV Codes": {"rich_text": [{"text": {"content": _safe_str(contract.get("cpv_codes"))[:2000]}}]},
        "Client": {"rich_text": [{"text": {"content": _safe_str(contract.get("client"), "Not Disclosed")[:2000]}}]},
        "Contract Link": {"url": _safe_str(contract.get("link")) or None},
        "Date Added": {"date": {"start": now_iso, "end": None}},
        "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
        "Description": {"rich_text": [{"text": {"content": _safe_str(contract.get("description"), "Not Disclosed")[:2000]}}]},
        "Employer Website": {"url": _safe_str(contract.get("client_link")) or None},
        "Language": {"rich_text": [{"text": {"content": _safe_str(contract.get("language"))[:2000]}}]},
        "Location": {"rich_text": [{"text": {"content": _safe_str(contract.get("country"))[:2000]}}]},
        "Reviewed By": {"select": {"name": "N/A"}},
        "Review Status": {"select": {"name": "Not Reviewed"}},
        "Value": {"rich_text": [{"text": {"content": _safe_str(contract.get("value"), "Unavailable")[:2000]}}]},
        "Contract Status": {"select": {"name": "Open"}},
        "Source": {"select": {"name": "EU Commission"}},
    }

    try:
        success = create_page(props)
        if success:
            new_titles_for_csv.append({"Title": name})
        else:
            print(f"⚠️ Failed to upload, not recording in dedup CSV: {name}")
    except Exception as e:
        print(f"Error creating Notion page for '{name}': {e}")

# Save newly uploaded titles to CSV for next run's dedup
if new_titles_for_csv:
    new_df = pd.DataFrame(new_titles_for_csv, columns=["Title"])
    header_needed = not os.path.exists(csv_path)
    new_df.to_csv(csv_path, mode="a", header=header_needed, index=False)

print(f"\u2705 Uploaded {len(new_titles_for_csv)} new EU Commission contracts to Notion.")


✅ Uploaded 0 new EU Commission contracts to Notion.
